# 01 · Recopilación de datos con la API REST de SpaceX
**Proyecto final – Capstone de Ciencia de Datos Aplicada (IBM)** · Autora: Jhoselyn Vera

**Objetivo:** obtener los lanzamientos del Falcon 9 desde la API de SpaceX y construir un DataFrame con
especificaciones del cohete, carga útil, órbita, sitio de lanzamiento y resultado de aterrizaje.

Por defecto se usa una **respuesta estática** de la API (`USE_LIVE_API = False`) para que los resultados
coincidan con el conjunto de datos del curso. Con `True` se consulta la API en vivo (incluye lanzamientos más recientes).

In [ ]:
import datetime
from functools import lru_cache

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", None)

In [ ]:
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
USE_LIVE_API = False
STATIC_JSON = BASE + "API_call_spacex_api.json"
LIVE_URL = "https://api.spacexdata.com/v4/launches/past"

url = LIVE_URL if USE_LIVE_API else STATIC_JSON
response = requests.get(url, timeout=60)
response.raise_for_status()
print("Código de estado:", response.status_code)
data = pd.json_normalize(response.json())
data[["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]].head()

## Limpieza inicial: lanzamientos con un solo núcleo y una sola carga útil

In [ ]:
data = data[["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]]
data = data[data["cores"].map(len) == 1]
data = data[data["payloads"].map(len) == 1]
data["cores"] = data["cores"].map(lambda x: x[0])
data["payloads"] = data["payloads"].map(lambda x: x[0])
data["date"] = pd.to_datetime(data["date_utc"]).dt.date
data = data[data["date"] <= datetime.date(2020, 11, 13)]  # fecha de corte del curso
print("Lanzamientos:", len(data))

## Consultas a la API por identificador (cohete, plataforma, carga útil y núcleo)

In [ ]:
@lru_cache(maxsize=None)
def get_json(endpoint, item_id):
    r = requests.get(f"https://api.spacexdata.com/v4/{endpoint}/{item_id}", timeout=60)
    r.raise_for_status()
    return r.json()

rows = []
for _, launch in data.iterrows():
    rocket = get_json("rockets", launch["rocket"])
    pad = get_json("launchpads", launch["launchpad"])
    payload = get_json("payloads", launch["payloads"])
    core_launch = launch["cores"]                      # información del núcleo en este vuelo
    core = get_json("cores", core_launch["core"]) if core_launch["core"] else {}
    rows.append({
        "FlightNumber": launch["flight_number"],
        "Date": launch["date"],
        "BoosterVersion": rocket["name"],
        "PayloadMass": payload["mass_kg"],
        "Orbit": payload["orbit"],
        "LaunchSite": pad["name"],
        "Outcome": f"{core_launch['landing_success']} {core_launch['landing_type']}",
        "Flights": core_launch["flight"],
        "GridFins": core_launch["gridfins"],
        "Reused": core_launch["reused"],
        "Legs": core_launch["legs"],
        "LandingPad": core_launch["landpad"],
        "Block": core.get("block"),
        "ReusedCount": core.get("reuse_count"),
        "Serial": core.get("serial"),
        "Longitude": pad["longitude"],
        "Latitude": pad["latitude"],
    })

launch_df = pd.DataFrame(rows)
launch_df.head()

## Filtrar solo Falcon 9 y tratar valores faltantes

In [ ]:
data_falcon9 = launch_df[launch_df["BoosterVersion"] != "Falcon 1"].copy()
data_falcon9["FlightNumber"] = list(range(1, data_falcon9.shape[0] + 1))

print("Valores nulos antes:")
print(data_falcon9.isnull().sum()[lambda s: s > 0])

mean_mass = data_falcon9["PayloadMass"].mean()
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].replace(np.nan, mean_mass)

data_falcon9.to_csv("../data/dataset_part_1.csv", index=False)
print("Filas:", len(data_falcon9), "· guardado en data/dataset_part_1.csv")
data_falcon9.head()